In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
os.getcwd()

'/Users/timmrahrt/billing_project'

# 0. Functions

In [3]:
# Helper functions
def read_file(filepath):
    '''
    Helper to load .xlsx and .csv files. Throws an error if any other type is being uploaded.
    '''
    file, extention = os.path.splitext(filepath)
    try:
        if extention == '.csv':
            file = pd.read_csv(filepath)
        
        elif extention == '.xlsx':
            file = pd.read_excel(filepath)
        
        else: 
            raise ValueError("Wrong filetype. Please upload .xlsx or .csv")
               
        return file
    
    except Exception as e:
        print(f"Error reading file {filepath}: {e}")


In [4]:
def load_data(*billing_files, shipment_raw):
    '''
    Loads all billing files and shipment report, concatenates and merges them.
    Accepts any number of billing files.
    '''
    sheets = []
    for file in billing_files:
        sheets.append(read_file(file))
    
    df = pd.concat(sheets, ignore_index=True)
    shipment_df = read_file(shipment_raw)
    data = pd.merge(df, shipment_df, how='left', left_on='Job', right_on='Shipment Job')
    
    return data

In [5]:
data = load_data(
    'Files/CargoWise Export - 20260511160204 - 197.xlsx',
    'Files/CargoWise Export - 20260511160347 - 284.xlsx',
    shipment_raw='Files/shipment_report_test.xlsx'
)

In [6]:
def build_shipment_data(data):
    '''
    Builds clean shipment dataset for the excel workbook.
    '''
    client_df = data[['Shipment Job', 'Supplier Name', 'Loading Port', 'Destination Port', 'Order Reference', 'Incoterms', 'Containers', 'Container Count', 
                    'House Bill', 'Mode', 'Vessel / Voyage', 'ETD', 'ETA', "Weight (KG)", "Volume (M3)", "Packages", "Package Unit"]]
    client_df = client_df.sort_values(by='Shipment Job').drop_duplicates('Shipment Job')
    
    return client_df

In [7]:
def build_billing_summary(data):
    '''
    Builds clean billing summary dataset.
    '''
    
    client_df = build_shipment_data(data)
    
    # AUD charges only
    aud = data[data['Currency'] == 'AUD'].groupby('Shipment Job')['Total'].sum().reset_index().rename(columns={'Total': 'AUD'})

    usd = data[data['Currency'] == 'USD'].groupby('Shipment Job')['Total'].sum().reset_index().rename(columns={'Total': 'USD'})
    # USD charges only

    # Local Total across ALL rows (AUD equivalent of everything)
    local = data.groupby('Shipment Job')['Local Total'].sum().reset_index().rename(columns={'Local Total': 'Local Total (AUD)'})

    # Merge the three together
    currency_data = pd.merge(aud, usd, how='outer', on='Shipment Job')
    currency_data = pd.merge(currency_data, local, how='left', on='Shipment Job')
    currency_data = currency_data.fillna(0)

    # Final job summary
    job_summary = pd.merge(client_df, currency_data, how='left', on='Shipment Job')
    billing_sum = job_summary.drop(columns=['House Bill','Mode', 'Weight (KG)', 'Volume (M3)', 'Packages','Package Unit', 'ETD', "ETA", "Container Count", "Vessel / Voyage"])
    billing_sum
    
    return billing_sum

In [8]:
def build_billing_detail(data):
    '''
    Builds a clean detailed billing sheet
    '''
    
    billing_detail = data[["Shipment Job", "Order Reference", "Supplier Name", "Loading Port", "Destination Port", "Incoterms", "Description", "Currency", "Amount", "Tax", "Total", "Local Total", "Exchange Rate", "Containers"]]
    
    return billing_detail

In [9]:
def build_supplier_summary(data):
    '''
    Builds a clean summary overview sheet.
    '''
    
    billing_detail = build_billing_detail(data)
    client_df = build_shipment_data(data)
    
    supplier_summary = billing_detail.groupby('Supplier Name').agg(
        Shipments=('Shipment Job', 'nunique'),
        Charge_lines=('Shipment Job', 'count'),
        Local_Total_AUD=('Local Total', 'sum')
    ).reset_index()

    supplier_summary = pd.merge(
        supplier_summary,
        client_df[['Supplier Name', 'Container Count']].groupby('Supplier Name')['Container Count'].sum().reset_index(),
        how='left',
        on='Supplier Name'
    )

    supplier_summary = supplier_summary.sort_values('Local_Total_AUD', ascending=False).reset_index(drop=True)
    
    return supplier_summary


## 1. Getting the data
### 1.1 CW_Export 1

In [52]:
# First datasheet
cw_export_aud = pd.read_excel("Files/CargoWise Export - 20260511160204 - 197.xlsx")
cw_export_aud.head()

,Charges,Charge Type,Job,Description,Branch,Dept,Currency,Exchange Rate,Amount,Tax ID,...,Supply Type,Tax Basis,Local Client,Overseas Agent,Sub Account 1,Sub Account 1 Type,Sub Account 2,Sub Account 2 Type,Branch Name,Department Description
0,ARB,MRG,B00527507,Technology fee,SYD,CIS,AUD,1,5.14,FREEGST,...,NaN,Accrual,GREENSYD,NaN,NaN,NaN,NaN,NaN,Sydney,Clearance Import Sea
1,ARB,MRG,B00530546,Technology Fee,SYD,CIS,AUD,1,5.14,FREEGST,...,NaN,Accrual,GREENSYD,NaN,NaN,NaN,NaN,NaN,Sydney,Clearance Import Sea
2,BDOF,MRG,B00527949,Delivery Order Fee,SYD,CIS,AUD,1,45.00,GST,...,NaN,Accrual,GREENSYD,NaN,NaN,NaN,NaN,NaN,Sydney,Clearance Import Sea
3,BDOF,MRG,B00527507,Delivery Order Fee,SYD,CIS,AUD,1,45.00,GST,...,NaN,Accrual,GREENSYD,NaN,NaN,NaN,NaN,NaN,Sydney,Clearance Import Sea
4,BDOF,MRG,B00523020,Delivery Order Fee,SYD,CIS,AUD,1,45.00,GST,...,NaN,Accrual,GREENSYD,NaN,NaN,NaN,NaN,NaN,Sydney,Clearance Import Sea


In [53]:
print("Shape:")
print(cw_export_aud.shape,"\n\n")
print("Columns:")
print(cw_export_aud.columns,"\n\n")
print("Data Types:")
print(cw_export_aud.dtypes,"\n\n")
print("Missing data:")
print(cw_export_aud.isna().sum(),"\n\n")

Shape:
(150, 31) 


Columns:
Index(['Charges', 'Charge Type', 'Job', 'Description', 'Branch', 'Dept',
       'Currency', 'Exchange Rate', 'Amount', 'Tax ID', 'Tax Date', 'Tax',
       'Total', 'Local Total', 'Local Amount', 'Local Tax', 'Prevent Grouping',
       'GL Account', 'Sequence', 'Job Local Ref', 'Tax Msg.', 'Supply Type',
       'Tax Basis', 'Local Client', 'Overseas Agent', 'Sub Account 1',
       'Sub Account 1 Type', 'Sub Account 2', 'Sub Account 2 Type',
       'Branch Name', 'Department Description'],
      dtype='str') 


Data Types:
Charges                              str
Charge Type                          str
Job                                  str
Description                          str
Branch                               str
Dept                                 str
Currency                             str
Exchange Rate                      int64
Amount                           float64
Tax ID                               str
Tax Date                  datetime

### CW_Export 2

In [54]:
# Second datasheet
cw_export_usd = pd.read_excel("Files/CargoWise Export - 20260511160347 - 284.xlsx")
cw_export_usd.head()

,Charges,Charge Type,Job,Description,Branch,Dept,Currency,Exchange Rate,Amount,Tax ID,...,Supply Type,Tax Basis,Local Client,Overseas Agent,Sub Account 1,Sub Account 1 Type,Sub Account 2,Sub Account 2 Type,Branch Name,Department Description
0,FRT,MRG,S04961296,International Freight - 1 40HC Container(s) @ ...,SYD,FIS,USD,0.685401,1700.0,FREEGST,...,NaN,Accrual,GREENSYD,ROHTHABKK,NaN,NaN,NaN,NaN,Sydney,Forwarding Import Sea
1,FRT,MRG,S04963081,International Freight - 1 20GP Container(s) @ ...,SYD,FIS,USD,0.684700,850.0,FREEGST,...,NaN,Accrual,GREENSYD,ROHCHICAN,NaN,NaN,NaN,NaN,Sydney,Forwarding Import Sea
2,FRT,MRG,S04968673,International Freight - 1 40HC Container(s) @ ...,SYD,FIS,USD,0.692101,1700.0,FREEGST,...,NaN,Accrual,GREENSYD,ROHTHABKK,NaN,NaN,NaN,NaN,Sydney,Forwarding Import Sea
3,FRT,MRG,S04968847,International Freight - 1 40HC Container(s) @ ...,SYD,FIS,USD,0.690700,1700.0,FREEGST,...,NaN,Accrual,GREENSYD,ROHTHABKK,NaN,NaN,NaN,NaN,Sydney,Forwarding Import Sea
4,FRT,MRG,S04961194,International Freight - 3 20GP Container(s) @ ...,SYD,FIS,USD,0.685399,2550.0,FREEGST,...,NaN,Accrual,GREENSYD,ROHTHABKK,NaN,NaN,NaN,NaN,Sydney,Forwarding Import Sea


In [55]:
print("Shape:")
print(cw_export_usd.shape,"\n\n")
print("Columns:")
print(cw_export_usd.columns,"\n\n")
print("Data Types:")
print(cw_export_usd.dtypes,"\n\n")
print("Missing data:")
print(cw_export_usd.isna().sum(),"\n\n")

Shape:
(23, 31) 


Columns:
Index(['Charges', 'Charge Type', 'Job', 'Description', 'Branch', 'Dept',
       'Currency', 'Exchange Rate', 'Amount', 'Tax ID', 'Tax Date', 'Tax',
       'Total', 'Local Total', 'Local Amount', 'Local Tax', 'Prevent Grouping',
       'GL Account', 'Sequence', 'Job Local Ref', 'Tax Msg.', 'Supply Type',
       'Tax Basis', 'Local Client', 'Overseas Agent', 'Sub Account 1',
       'Sub Account 1 Type', 'Sub Account 2', 'Sub Account 2 Type',
       'Branch Name', 'Department Description'],
      dtype='str') 


Data Types:
Charges                              str
Charge Type                          str
Job                                  str
Description                          str
Branch                               str
Dept                                 str
Currency                             str
Exchange Rate                    float64
Amount                           float64
Tax ID                               str
Tax Date                  datetime6

In [56]:
print("Columns necessary? Tax Msg., Supply Type, Sub Account 1, Sub Account 1 Type, Sub Account 2, Sub Account 2 Type")

Columns necessary? Tax Msg., Supply Type, Sub Account 1, Sub Account 1 Type, Sub Account 2, Sub Account 2 Type


In [57]:
# Get the shipment test data
shipment_test = pd.read_excel("Files/shipment_report_test.xlsx")
shipment_test.head()

,Shipment Job,House Bill,Mode,Service Level,Ocean BOL,Booking Ref,Vessel / Voyage,Supplier Name,Loading Port,ETD,...,ETA,Order Reference,Incoterms,Containers,Container Count,Weight (KG),Volume (M3),Packages,Package Unit,Customs Entry
0,B00523020,RAF0202716,SEA FCL,Classic,BKK600287600,BKK600287600,MAERSK RIO DELTA / 615S,"GREENATURE GRAINS AND NOODLE CO., LTD.",THLKR,2026-03-15,...,2026-04-02,"54142, 54748",FOB,PIDU4568352,1,7010.20,68.210,9882,CTN,AFHK96FKA
1,B00527507,SGSYD5373006,SEA FCL,Classic,ONEYSING19641900,ONEYSING19641900,ONE OLYMPUS / 108S,TAT HUI FOODS PTE LTD,SGSIN,2026-04-11,...,2026-04-26,54465,FOB,"GAOU6609000, TCLU1797622",2,17953.00,112.611,4461,CTN,AFHR7TRRT
2,S04933292,SMEZF26000261,SEA FCL,Classic,265636062,265636062,MAERSK RIO DELTA / 615S,DIJO BAKING SP.Z O.O.,PLWRO,2026-02-10,...,2026-04-27,54284,DAP,MRSU8758354:40HC,1,8967.60,8.968,5640,COI,AFHR3LKRY
3,S04961296,S04961296,SEA FCL,Sea - Named Accounts,OOLU2167262210,2167262210,OOCL YOKOHAMA / 210S,"OCTA FOODS CO., LTD",THLKR,2026-04-12,...,2026-04-28,54228,DAP,FFAU6148739:40HC,1,21997.91,64.510,6451,CTN,AFHR7HNF7
4,S04963081,S04963081,SEA FCL,Classic,266970733,266970733,MAERSK RIO DELTA / 615S,"GUANGZHOU LIANYI BIOTECHNOLOGY CO.,LTD",CNNSA,2026-04-01,...,2026-04-28,54289A,FCA,MRKU7238735:20GP,1,20920.15,28.290,5893,CTN,AFHTCKMP9


## 2. Data manipulation

In [58]:
print(cw_export_aud.value_counts('Job'))
print('\n\n')
print(cw_export_usd.value_counts('Job'))

Job
S04968847    13
S04963081    13
S04961194    13
S04977389    13
B00527507    12
B00527949    12
B00523020    12
B00528168    12
S04954581    12
S04933292    12
B00530546    11
S04961296     8
S04968673     7
Name: count, dtype: int64



Job
S04954581    9
S04933292    8
S04961296    1
S04963081    1
S04968673    1
S04968847    1
S04961194    1
S04977389    1
Name: count, dtype: int64


In [59]:
# Merge the two datasets
df = pd.concat([cw_export_aud,cw_export_usd], ignore_index=True)
data = pd.merge(df, shipment_test, how='left', left_on="Job", right_on="Shipment Job")
data.head()

,Charges,Charge Type,Job,Description,Branch,Dept,Currency,Exchange Rate,Amount,Tax ID,...,ETA,Order Reference,Incoterms,Containers,Container Count,Weight (KG),Volume (M3),Packages,Package Unit,Customs Entry
0,ARB,MRG,B00527507,Technology fee,SYD,CIS,AUD,1.0,5.14,FREEGST,...,2026-04-26,54465,FOB,"GAOU6609000, TCLU1797622",2,17953.00,112.611,4461,CTN,AFHR7TRRT
1,ARB,MRG,B00530546,Technology Fee,SYD,CIS,AUD,1.0,5.14,FREEGST,...,2026-05-09,"55162, 55163",CIF,GCXU5824762,1,19218.89,50.000,22,PLT,AFHXLFTAF
2,BDOF,MRG,B00527949,Delivery Order Fee,SYD,CIS,AUD,1.0,45.00,GST,...,2026-05-01,"54752, 54753",DAP,"PIDU4331340, PIDU4430321",2,11903.00,140.780,9460,CTN,AFHWKGCXK
3,BDOF,MRG,B00527507,Delivery Order Fee,SYD,CIS,AUD,1.0,45.00,GST,...,2026-04-26,54465,FOB,"GAOU6609000, TCLU1797622",2,17953.00,112.611,4461,CTN,AFHR7TRRT
4,BDOF,MRG,B00523020,Delivery Order Fee,SYD,CIS,AUD,1.0,45.00,GST,...,2026-04-02,"54142, 54748",FOB,PIDU4568352,1,7010.20,68.210,9882,CTN,AFHK96FKA


In [60]:
# Missing data introduced?
data.isna().sum()

Charges                     0
Charge Type                 0
Job                         0
Description                 0
Branch                      0
Dept                        0
Currency                    0
Exchange Rate               0
Amount                      0
Tax ID                      0
Tax Date                    0
Tax                         0
Total                       0
Local Total                 0
Local Amount                0
Local Tax                   0
Prevent Grouping            0
GL Account                  0
Sequence                    0
Job Local Ref               0
Tax Msg.                  173
Supply Type               173
Tax Basis                   0
Local Client                0
Overseas Agent             59
Sub Account 1             173
Sub Account 1 Type        173
Sub Account 2             173
Sub Account 2 Type        173
Branch Name                 0
Department Description      0
Shipment Job                0
House Bill                  0
Mode      

In [61]:
# Column check
data.columns

Index(['Charges', 'Charge Type', 'Job', 'Description', 'Branch', 'Dept',
       'Currency', 'Exchange Rate', 'Amount', 'Tax ID', 'Tax Date', 'Tax',
       'Total', 'Local Total', 'Local Amount', 'Local Tax', 'Prevent Grouping',
       'GL Account', 'Sequence', 'Job Local Ref', 'Tax Msg.', 'Supply Type',
       'Tax Basis', 'Local Client', 'Overseas Agent', 'Sub Account 1',
       'Sub Account 1 Type', 'Sub Account 2', 'Sub Account 2 Type',
       'Branch Name', 'Department Description', 'Shipment Job', 'House Bill',
       'Mode', 'Service Level', 'Ocean BOL', 'Booking Ref', 'Vessel / Voyage',
       'Supplier Name', 'Loading Port', 'ETD', 'Destination Port', 'ETA',
       'Order Reference', 'Incoterms', 'Containers', 'Container Count',
       'Weight (KG)', 'Volume (M3)', 'Packages', 'Package Unit',
       'Customs Entry'],
      dtype='str')

## 3. Create subsets
### 3.1 Client-specific dataset

In [ ]:
# Sheet 1
client_df = data[['Shipment Job', 'Supplier Name', 'Loading Port', 'Destination Port', 'Order Reference', 'Incoterms', 'Containers', 'Container Count', 
                  'House Bill', 'Mode', 'Vessel / Voyage', 'ETD', 'ETA', "Weight (KG)", "Volume (M3)", "Packages", "Package Unit"]]
client_df = client_df.sort_values(by='Shipment Job').drop_duplicates('Shipment Job')
client_df

,Shipment Job,Supplier Name,Loading Port,Destination Port,Order Reference,Incoterms,Containers,Container Count,House Bill,Mode,Vessel / Voyage,ETD,ETA,Weight (KG),Volume (M3),Packages,Package Unit
4,B00523020,"GREENATURE GRAINS AND NOODLE CO., LTD.",THLKR,AUSYD,"54142, 54748",FOB,PIDU4568352,1,RAF0202716,SEA FCL,MAERSK RIO DELTA / 615S,2026-03-15,2026-04-02,7010.20,68.210,9882,CTN
0,B00527507,TAT HUI FOODS PTE LTD,SGSIN,AUSYD,54465,FOB,"GAOU6609000, TCLU1797622",2,SGSYD5373006,SEA FCL,ONE OLYMPUS / 108S,2026-04-11,2026-04-26,17953.00,112.611,4461,CTN
2,B00527949,"GREENATURE GRAINS AND NOODLE CO., LTD.",THLCH,AUSYD,"54752, 54753",DAP,"PIDU4331340, PIDU4430321",2,RAF0233484,SEA FCL,OOCL YOKOHAMA / 210S,2026-04-12,2026-05-01,11903.00,140.780,9460,CTN
5,B00528168,"GREENATURE GRAINS AND NOODLE CO., LTD.",THBKK,AUSYD,54751,DAP,BEAU5473209,1,RAF0233483,SEA FCL,ONE CONTINUITY / 042S,2026-04-02,2026-05-05,6321.00,73.500,14700,CTN
1,B00530546,RONDO FOOD GMBH & CO KG,ITSPE,AUSYD,"55162, 55163",CIF,GCXU5824762,1,SYD5354270,SEA FCL,MAERSK EMDEN / 412S,2026-03-22,2026-05-09,19218.89,50.000,22,PLT
23,S04933292,DIJO BAKING SP.Z O.O.,PLWRO,AUSYD,54284,DAP,MRSU8758354:40HC,1,SMEZF26000261,SEA FCL,MAERSK RIO DELTA / 615S,2026-02-10,2026-04-27,8967.60,8.968,5640,COI
16,S04954581,DIJO BAKING SP.Z O.O.,PLWRO,AUSYD,54285,DAP,MRSU7492700:40HC,1,SMEZF26001049,SEA FCL,MAERSK RIO ALFA / 616S,2026-02-27,2026-05-03,7707.51,7707.510,4263,PKG
18,S04961194,"C.P. INTERTRADE CO.,LTD.",THLKR,AUSYD,54505,FCA,"CSNU1625899:20GP, CSNU2861517:20GP, CSNU299883...",3,S04961194,SEA FCL,OOCL YOKOHAMA / 210S,2026-04-12,2026-05-02,62020.00,75.000,7000,CTN
12,S04961296,"OCTA FOODS CO., LTD",THLKR,AUMEL,54228,DAP,FFAU6148739:40HC,1,S04961296,SEA FCL,OOCL YOKOHAMA / 210S,2026-04-12,2026-04-28,21997.91,64.510,6451,CTN
17,S04963081,"GUANGZHOU LIANYI BIOTECHNOLOGY CO.,LTD",CNNSA,AUSYD,54289A,FCA,MRKU7238735:20GP,1,S04963081,SEA FCL,MAERSK RIO DELTA / 615S,2026-04-01,2026-04-28,20920.15,28.290,5893,CTN


In [64]:
# Final sanity check
client_df.isna().sum()

Shipment Job        0
Supplier Name       0
Loading Port        0
Destination Port    0
Order Reference     0
Incoterms           0
Containers          0
Container Count     0
House Bill          0
Mode                0
Vessel / Voyage     0
ETD                 0
ETA                 0
Weight (KG)         0
Volume (M3)         0
Packages            0
Package Unit        0
dtype: int64

### 3.2 Billing Summary

In [ ]:
# AUD charges only
aud = data[data['Currency'] == 'AUD'].groupby('Shipment Job')['Total'].sum().reset_index().rename(columns={'Total': 'AUD'})

# USD charges only
usd = data[data['Currency'] == 'USD'].groupby('Shipment Job')['Total'].sum().reset_index().rename(columns={'Total': 'USD'})

# Local Total across ALL rows (AUD equivalent of everything)
local = data.groupby('Shipment Job')['Local Total'].sum().reset_index().rename(columns={'Local Total': 'Local Total (AUD)'})

# Merge the three together
currency_data = pd.merge(aud, usd, how='outer', on='Shipment Job')
currency_data = pd.merge(currency_data, local, how='left', on='Shipment Job')
currency_data = currency_data.fillna(0)

# Final job summary
job_summary = pd.merge(client_df, currency_data, how='left', on='Shipment Job')
billing_sum = job_summary.drop(columns=['House Bill','Mode', 'Weight (KG)', 'Volume (M3)', 'Packages','Package Unit', 'ETD', "ETA", "Container Count", "Vessel / Voyage"])
billing_sum

,Shipment Job,Supplier Name,Loading Port,Destination Port,Order Reference,Incoterms,Containers,Container Count,House Bill,Mode,Vessel / Voyage,ETD,ETA,Weight (KG),Volume (M3),Packages,Package Unit,AUD,USD,Local Total (AUD)
0,B00523020,"GREENATURE GRAINS AND NOODLE CO., LTD.",THLKR,AUSYD,"54142, 54748",FOB,PIDU4568352,1,RAF0202716,SEA FCL,MAERSK RIO DELTA / 615S,2026-03-15,2026-04-02,7010.20,68.210,9882,CTN,3634.29,0.00,3634.29
1,B00527507,TAT HUI FOODS PTE LTD,SGSIN,AUSYD,54465,FOB,"GAOU6609000, TCLU1797622",2,SGSYD5373006,SEA FCL,ONE OLYMPUS / 108S,2026-04-11,2026-04-26,17953.00,112.611,4461,CTN,6949.44,0.00,6949.44
2,B00527949,"GREENATURE GRAINS AND NOODLE CO., LTD.",THLCH,AUSYD,"54752, 54753",DAP,"PIDU4331340, PIDU4430321",2,RAF0233484,SEA FCL,OOCL YOKOHAMA / 210S,2026-04-12,2026-05-01,11903.00,140.780,9460,CTN,6552.37,0.00,6552.37
3,B00528168,"GREENATURE GRAINS AND NOODLE CO., LTD.",THBKK,AUSYD,54751,DAP,BEAU5473209,1,RAF0233483,SEA FCL,ONE CONTINUITY / 042S,2026-04-02,2026-05-05,6321.00,73.500,14700,CTN,3575.39,0.00,3575.39
4,B00530546,RONDO FOOD GMBH & CO KG,ITSPE,AUSYD,"55162, 55163",CIF,GCXU5824762,1,SYD5354270,SEA FCL,MAERSK EMDEN / 412S,2026-03-22,2026-05-09,19218.89,50.000,22,PLT,3370.98,0.00,3370.98
5,S04933292,DIJO BAKING SP.Z O.O.,PLWRO,AUSYD,54284,DAP,MRSU8758354:40HC,1,SMEZF26000261,SEA FCL,MAERSK RIO DELTA / 615S,2026-02-10,2026-04-27,8967.60,8.968,5640,COI,3397.90,2177.00,6570.92
6,S04954581,DIJO BAKING SP.Z O.O.,PLWRO,AUSYD,54285,DAP,MRSU7492700:40HC,1,SMEZF26001049,SEA FCL,MAERSK RIO ALFA / 616S,2026-02-27,2026-05-03,7707.51,7707.510,4263,PKG,3240.99,2487.59,6835.23
7,S04961194,"C.P. INTERTRADE CO.,LTD.",THLKR,AUSYD,54505,FCA,"CSNU1625899:20GP, CSNU2861517:20GP, CSNU299883...",3,S04961194,SEA FCL,OOCL YOKOHAMA / 210S,2026-04-12,2026-05-02,62020.00,75.000,7000,CTN,7927.45,2550.00,11647.91
8,S04961296,"OCTA FOODS CO., LTD",THLKR,AUMEL,54228,DAP,FFAU6148739:40HC,1,S04961296,SEA FCL,OOCL YOKOHAMA / 210S,2026-04-12,2026-04-28,21997.91,64.510,6451,CTN,1585.50,1700.00,4065.80
9,S04963081,"GUANGZHOU LIANYI BIOTECHNOLOGY CO.,LTD",CNNSA,AUSYD,54289A,FCA,MRKU7238735:20GP,1,S04963081,SEA FCL,MAERSK RIO DELTA / 615S,2026-04-01,2026-04-28,20920.15,28.290,5893,CTN,3048.50,850.00,4289.92


### 3.3 Billing Detail

In [67]:
billing_detail = data[["Shipment Job", "Order Reference", "Supplier Name", "Loading Port", "Destination Port", "Incoterms", "Description", "Currency", "Amount", "Tax", "Total", "Local Total", "Exchange Rate", "Containers"]]
billing_detail

,Shipment Job,Order Reference,Supplier Name,Loading Port,Destination Port,Incoterms,Description,Currency,Amount,Tax,Total,Local Total,Exchange Rate,Containers
0,B00527507,54465,TAT HUI FOODS PTE LTD,SGSIN,AUSYD,FOB,Technology fee,AUD,5.14,0.0,5.14,5.14,1.000000,"GAOU6609000, TCLU1797622"
1,B00530546,"55162, 55163",RONDO FOOD GMBH & CO KG,ITSPE,AUSYD,CIF,Technology Fee,AUD,5.14,0.0,5.14,5.14,1.000000,GCXU5824762
2,B00527949,"54752, 54753","GREENATURE GRAINS AND NOODLE CO., LTD.",THLCH,AUSYD,DAP,Delivery Order Fee,AUD,45.00,4.5,49.50,49.50,1.000000,"PIDU4331340, PIDU4430321"
3,B00527507,54465,TAT HUI FOODS PTE LTD,SGSIN,AUSYD,FOB,Delivery Order Fee,AUD,45.00,4.5,49.50,49.50,1.000000,"GAOU6609000, TCLU1797622"
4,B00523020,"54142, 54748","GREENATURE GRAINS AND NOODLE CO., LTD.",THLKR,AUSYD,FOB,Delivery Order Fee,AUD,45.00,4.5,49.50,49.50,1.000000,PIDU4568352
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,S04954581,54285,DIJO BAKING SP.Z O.O.,PLWRO,AUSYD,DAP,Port Security Fee - Origin - 1 Container(s) @ ...,USD,12.00,0.0,12.00,17.34,0.692042,MRSU7492700:40HC
169,S04933292,54284,DIJO BAKING SP.Z O.O.,PLWRO,AUSYD,DAP,Port Security Fee - Origin - 1 Container(s) @ ...,USD,12.00,0.0,12.00,17.49,0.686106,MRSU8758354:40HC
170,S04933292,54284,DIJO BAKING SP.Z O.O.,PLWRO,AUSYD,DAP,SOLAS VGM Processing / Administration Fee - 1 ...,USD,25.00,0.0,25.00,36.44,0.686059,MRSU8758354:40HC
171,S04954581,54285,DIJO BAKING SP.Z O.O.,PLWRO,AUSYD,DAP,SOLAS VGM Processing / Administration Fee - 1 ...,USD,25.00,0.0,25.00,36.12,0.692137,MRSU7492700:40HC


### 3.4 Supplier Summary

In [9]:
supplier_summary = billing_detail.groupby('Supplier Name').agg(
    Shipments = ("Shipment Job", "nunique"),
    Charge_lines = ("Shipment Job", "count"),
    Local_Total_AUD = ("Local Total", "sum")
).reset_index()

supplier_summary = pd.merge(
    supplier_summary,
    client_df[['Supplier Name', 'Container Count']].groupby('Supplier Name')['Container Count'].sum().reset_index(),
    how='left',
    on='Supplier Name'
)

supplier_summary = supplier_summary.sort_values('Local_Total_AUD', ascending=False).reset_index(drop=True)
supplier_summary


,Supplier Name,Shipments,Charge_lines,Local_Total_AUD,Container Count
0,"GREENATURE GRAINS AND NOODLE CO., LTD.",3,36,13762.05,4
1,DIJO BAKING SP.Z O.O.,2,41,13406.15,2
2,"C.P. INTERTRADE CO.,LTD.",1,14,11647.91,3
3,"OCTA FOODS CO., LTD",2,23,9808.06,2
4,TAT HUI FOODS PTE LTD,1,12,6949.44,2
5,TASTEL FINE FOOD PVT LTD,1,14,5480.49,1
6,"GUANGZHOU LIANYI BIOTECHNOLOGY CO.,LTD",1,14,4289.92,1
7,"SINGHA KAMEDA (THAILAND) CO., LTD.",1,8,4001.79,1
8,RONDO FOOD GMBH & CO KG,1,11,3370.98,1


In [72]:
## 4. Building the excel output
with pd.ExcelWriter('TGG_report.xlsx', engine='openpyxl') as writer:
    client_df.to_excel(writer, sheet_name="Shipment Summary", index=False)
    billing_sum.to_excel(writer, sheet_name="Billing Summary", index=False)
    billing_detail.to_excel(writer, sheet_name="Billing Detail", index=False)
    supplier_summary.to_excel(writer, sheet_name="Supplier Summary", index=False)